# LeetCode 977 — Squares of a Sorted Array bằng Python

**Link đề:** https://leetcode.com/problems/squares-of-a-sorted-array/
**Độ khó:** Easy · **Chủ đề:** Array, Two Pointers, Sorting

```python
class Solution:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu vì sao bình phương làm **mất** thứ tự của mảng đã sắp xếp (do số âm).
2. Ôn **list comprehension**, `sorted`, tạo mảng kích thước cố định `[0] * n`.
3. Nắm kỹ thuật **hai con trỏ từ hai đầu** (two pointers) và **điền kết quả từ cuối** để đạt O(n) — đúng yêu cầu *follow up* của đề.
4. Biết thêm: tìm điểm đổi dấu + **trộn** (merge) hai dãy, và `heapq.merge`.
5. Trace, chứng minh, phân tích Time/Space, và làm 6 bài tập tăng dần.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `nums` đã sắp **không giảm** (non-decreasing). Trả về mảng **bình phương** của từng phần tử, cũng sắp không giảm.

| nums | Bình phương (chưa sắp) | Kết quả |
|---|---|---|
| `[-4,-1,0,3,10]` | `[16,1,0,9,100]` | `[0,1,9,16,100]` |
| `[-7,-3,2,3,11]` | `[49,9,4,9,121]` | `[4,9,9,49,121]` |

**Constraints:** `1 <= len(nums) <= 10^4`, `-10^4 <= nums[i] <= 10^4`, `nums` đã sắp không giảm.

**Follow up:** bình phương rồi sắp xếp là cách tầm thường (O(n log n)). Hãy tìm lời giải **O(n)**.

## 2. Nền tảng Python

### 2.1. List comprehension

`[x * x for x in nums]` tạo list mới, mỗi phần tử là `x * x`. Tương đương vòng `for` + `append` nhưng gọn hơn.

In [1]:
nums = [-4, -1, 0, 3, 10]
squares = []
for x in nums:
    squares.append(x * x)
print("vòng for     :", squares)
print("comprehension:", [x * x for x in nums])
print("x ** 2 cũng được:", [x ** 2 for x in nums])

vòng for     : [16, 1, 0, 9, 100]
comprehension: [16, 1, 0, 9, 100]
x ** 2 cũng được: [16, 1, 0, 9, 100]


### 2.2. `abs` và quan hệ với bình phương

`x * x == abs(x) * abs(x)`. Vì vậy **so sánh bình phương ⇔ so sánh trị tuyệt đối**: `a² < b²` ⇔ `|a| < |b|`. Ta sẽ so trị tuyệt đối khi dùng hai con trỏ.

In [2]:
for a, b_ in [(-4, 3), (-1, 0), (-7, 11)]:
    print(f"a={a}, b={b_}: a²<b²? {a*a < b_*b_} | |a|<|b|? {abs(a) < abs(b_)}")

a=-4, b=3: a²<b²? False | |a|<|b|? False
a=-1, b=0: a²<b²? False | |a|<|b|? False
a=-7, b=11: a²<b²? True | |a|<|b|? True


### 2.3. Tạo mảng cố định và gán theo chỉ số

`[0] * n` tạo list `n` số 0. Sau đó gán `res[k] = ...` vào vị trí bất kỳ — khác với `append` (chỉ thêm vào cuối). Ta sẽ điền mảng kết quả **từ phải sang trái**.

Chỉ số âm: `res[-1]` là phần tử cuối, `res[-2]` là kế cuối.

In [3]:
res = [0] * 5
res[4] = 100
res[3] = 16
print(res, "| res[-1] =", res[-1])

[0, 0, 0, 16, 100] | res[-1] = 100


## 3. Cách trực giác (baseline) — bình phương rồi sắp xếp

In [4]:
def sorted_squares_sort(nums: list[int]) -> list[int]:
    return sorted(x * x for x in nums)

print(sorted_squares_sort([-4, -1, 0, 3, 10]))
print(sorted_squares_sort([-7, -3, 2, 3, 11]))

[0, 1, 9, 16, 100]
[4, 9, 9, 49, 121]


`sorted(...)` nhận thẳng một **generator expression** (không cần ngoặc vuông). Độ phức tạp **O(n log n)** do sắp xếp, bộ nhớ O(n) cho kết quả.

## 4. Nút thắt

Ta **vứt bỏ** thông tin quý giá: `nums` đã sắp xếp sẵn! Sắp xếp lại từ đầu là thừa.

Quan sát bình phương của một mảng đã sắp:

In [5]:
nums = [-7, -5, -2, 0, 1, 4, 6]
print("nums   :", nums)
print("squares:", [x * x for x in nums])

nums   : [-7, -5, -2, 0, 1, 4, 6]
squares: [49, 25, 4, 0, 1, 16, 36]


Bình phương có hình **chữ V**: giảm dần ở phần âm, rồi tăng dần ở phần không âm. Giá trị **nhỏ nhất** nằm ở giữa (gần số 0), còn giá trị **lớn nhất** luôn nằm ở **một trong hai đầu**.

## 5. Insight — hai con trỏ ở hai đầu, điền từ cuối

- Phần tử có bình phương lớn nhất là `nums[0]` hoặc `nums[-1]` (cái nào có trị tuyệt đối lớn hơn).
- Đặt `left = 0`, `right = n − 1`. So `|nums[left]|` với `|nums[right]|`: cái lớn hơn → bình phương của nó là **giá trị lớn nhất còn lại** → điền vào vị trí **cuối cùng còn trống** của kết quả, rồi dịch con trỏ đó vào trong.
- Lặp đến khi hai con trỏ gặp nhau. Mỗi bước điền đúng 1 ô → **O(n)**.

Tại sao điền từ **cuối**? Vì từ hai đầu ta luôn biết chắc phần tử **lớn nhất**, chứ không biết phần tử nhỏ nhất (nó nằm ở giữa).

## 6. Lời giải chính — xây từng đoạn rồi ghép

**Đoạn 1 — khởi tạo:**
```python
n = len(nums)
res = [0] * n
left, right = 0, n - 1
pos = n - 1           # ô trống cuối cùng
```
**Đoạn 2 — vòng lặp** chạy khi `left <= right` (dấu `=` để xử lý phần tử cuối cùng còn lại).

**Đoạn 3 — chọn bên lớn hơn**, điền vào `res[pos]`, dịch con trỏ, giảm `pos`.

In [6]:
def sorted_squares(nums: list[int]) -> list[int]:
    n = len(nums)                                  # Đoạn 1
    res = [0] * n
    left, right = 0, n - 1
    pos = n - 1
    while left <= right:                           # Đoạn 2
        if abs(nums[left]) > abs(nums[right]):     # Đoạn 3
            res[pos] = nums[left] * nums[left]
            left += 1
        else:
            res[pos] = nums[right] * nums[right]
            right -= 1
        pos -= 1
    return res

print(sorted_squares([-4, -1, 0, 3, 10]))
print(sorted_squares([-7, -3, 2, 3, 11]))

[0, 1, 9, 16, 100]
[4, 9, 9, 49, 121]


## 7. Trace từng bước

`nums = [-4, -1, 0, 3, 10]`:

| Bước | left | right | nums[left] | nums[right] | Chọn | pos | res sau bước |
|:---:|:---:|:---:|:---:|:---:|:---:|:---:|---|
| 1 | 0 | 4 | -4 | 10 | phải (10) | 4 | `[0,0,0,0,100]` |
| 2 | 0 | 3 | -4 | 3 | trái (-4) | 3 | `[0,0,0,16,100]` |
| 3 | 1 | 3 | -1 | 3 | phải (3) | 2 | `[0,0,9,16,100]` |
| 4 | 1 | 2 | -1 | 0 | trái (-1) | 1 | `[0,1,9,16,100]` |
| 5 | 2 | 2 | 0 | 0 | phải (0) | 0 | `[0,1,9,16,100]` |

In [7]:
def trace_sorted_squares(nums: list[int]) -> list[int]:
    n = len(nums)
    res = [0] * n
    left, right, pos = 0, n - 1, n - 1
    step = 1
    while left <= right:
        L, R = nums[left], nums[right]
        if abs(L) > abs(R):
            res[pos] = L * L
            side = f"trái ({L})"
            left += 1
        else:
            res[pos] = R * R
            side = f"phải ({R})"
            right -= 1
        print(f"Bước {step}: |{L}| vs |{R}| -> chọn {side:10} điền res[{pos}] = {res[pos]:>3} | res = {res}")
        pos -= 1
        step += 1
    return res

trace_sorted_squares([-4, -1, 0, 3, 10])
print()
trace_sorted_squares([-7, -3, 2, 3, 11])

Bước 1: |-4| vs |10| -> chọn phải (10)  điền res[4] = 100 | res = [0, 0, 0, 0, 100]
Bước 2: |-4| vs |3| -> chọn trái (-4)  điền res[3] =  16 | res = [0, 0, 0, 16, 100]
Bước 3: |-1| vs |3| -> chọn phải (3)   điền res[2] =   9 | res = [0, 0, 9, 16, 100]
Bước 4: |-1| vs |0| -> chọn trái (-1)  điền res[1] =   1 | res = [0, 1, 9, 16, 100]
Bước 5: |0| vs |0| -> chọn phải (0)   điền res[0] =   0 | res = [0, 1, 9, 16, 100]

Bước 1: |-7| vs |11| -> chọn phải (11)  điền res[4] = 121 | res = [0, 0, 0, 0, 121]
Bước 2: |-7| vs |3| -> chọn trái (-7)  điền res[3] =  49 | res = [0, 0, 0, 49, 121]
Bước 3: |-3| vs |3| -> chọn phải (3)   điền res[2] =   9 | res = [0, 0, 9, 49, 121]
Bước 4: |-3| vs |2| -> chọn trái (-3)  điền res[1] =   9 | res = [0, 9, 9, 49, 121]
Bước 5: |2| vs |2| -> chọn phải (2)   điền res[0] =   4 | res = [4, 9, 9, 49, 121]


[4, 9, 9, 49, 121]

Ở ví dụ thứ hai, có lúc `|−3| == |3|`: nhánh `else` chọn bên phải trước — kết quả vẫn đúng vì hai bình phương bằng nhau.

## 8. Code nộp LeetCode

### Cách A — hai con trỏ từ hai đầu (O(n))

In [8]:
class Solution:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        n = len(nums)
        res = [0] * n
        left, right = 0, n - 1
        for pos in range(n - 1, -1, -1):
            if abs(nums[left]) > abs(nums[right]):
                res[pos] = nums[left] * nums[left]
                left += 1
            else:
                res[pos] = nums[right] * nums[right]
                right -= 1
        return res

print(Solution().sortedSquares([-4, -1, 0, 3, 10]))
print(Solution().sortedSquares([-7, -3, 2, 3, 11]))

[0, 1, 9, 16, 100]
[4, 9, 9, 49, 121]


Biến thể gọn hơn: thay `while` bằng `for pos in range(n - 1, -1, -1)` — mỗi vòng điền đúng một ô, nên chạy đúng `n` lần là đủ.

### Cách B — bình phương + sắp xếp (O(n log n))

In [9]:
class SolutionSort:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        return sorted(x * x for x in nums)

print(SolutionSort().sortedSquares([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


### Cách C — tìm điểm đổi dấu rồi trộn ra hai phía

Tìm `k` = chỉ số số không âm đầu tiên (dùng `bisect_left(nums, 0)` — tìm nhị phân). Khi đó:
- Phần âm `nums[0..k-1]` đọc **ngược** (`k−1, k−2, ...`) cho bình phương **tăng dần**.
- Phần không âm `nums[k..]` đọc xuôi cũng cho bình phương **tăng dần**.

Trộn hai dãy tăng dần (giống bước merge của merge sort), điền từ **đầu** kết quả.

In [10]:
from bisect import bisect_left
print("bisect_left([-4,-1,0,3,10], 0) =", bisect_left([-4, -1, 0, 3, 10], 0))
print("bisect_left([-7,-3,2,3,11], 0) =", bisect_left([-7, -3, 2, 3, 11], 0))
print("bisect_left([-3,-2,-1], 0) =", bisect_left([-3, -2, -1], 0), "(tất cả đều âm)")

bisect_left([-4,-1,0,3,10], 0) = 2
bisect_left([-7,-3,2,3,11], 0) = 2
bisect_left([-3,-2,-1], 0) = 3 (tất cả đều âm)


In [11]:
class SolutionMerge:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        k = bisect_left(nums, 0)
        i, j = k - 1, k                  # i đi sang trái (phần âm), j đi sang phải
        res = []
        while i >= 0 and j < len(nums):
            if -nums[i] < nums[j]:
                res.append(nums[i] * nums[i]); i -= 1
            else:
                res.append(nums[j] * nums[j]); j += 1
        while i >= 0:
            res.append(nums[i] * nums[i]); i -= 1
        while j < len(nums):
            res.append(nums[j] * nums[j]); j += 1
        return res

print(SolutionMerge().sortedSquares([-7, -3, 2, 3, 11]))
print(SolutionMerge().sortedSquares([-3, -2, -1]))

[4, 9, 9, 49, 121]
[1, 4, 9]


### Cách D — `heapq.merge` (thư viện chuẩn trộn sẵn)

`heapq.merge(a, b)` trộn **nhiều dãy đã sắp** thành một dãy sắp, lười (lazy) và O(n) cho 2 dãy.

In [12]:
import heapq

class SolutionHeapMerge:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        k = bisect_left(nums, 0)
        neg = (x * x for x in reversed(nums[:k]))   # tăng dần
        pos = (x * x for x in nums[k:])             # tăng dần
        return list(heapq.merge(neg, pos))

print(SolutionHeapMerge().sortedSquares([-4, -1, 0, 3, 10]))

[0, 1, 9, 16, 100]


## 9. Vì sao đúng? (Cách A)

**Invariant:** trước mỗi vòng, đoạn `nums[left..right]` chứa đúng các phần tử **chưa** được điền, và `res[pos+1..n-1]` chứa bình phương của các phần tử **đã** xử lý, sắp tăng dần, và mỗi giá trị trong đó **≥** mọi bình phương còn lại trong `nums[left..right]`.

**Giữ được invariant:** `nums[left..right]` là một đoạn con đã sắp, nên phần tử có trị tuyệt đối lớn nhất của đoạn nằm ở `left` hoặc `right` (hình chữ V). Ta chọn đúng phần tử đó → bình phương của nó là lớn nhất còn lại, và ≤ các giá trị đã điền → đặt vào `res[pos]` giữ thứ tự tăng.

**Kết thúc:** sau `n` bước, mọi ô đã được điền, `res` sắp tăng dần và là hoán vị của các bình phương.

## 10. Độ phức tạp và so sánh

| Cách | Time | Space (ngoài output) | Nhận xét |
|---|---|---|---|
| B. Bình phương + sort | O(n log n) | O(n) tạm cho sort | 1 dòng, chấp nhận được |
| **A. Hai con trỏ** | **O(n)** | **O(1)** | chuẩn follow-up |
| C. Bisect + merge | O(n) (+ O(log n) tìm k) | O(1) | trực quan "trộn hai dãy" |
| D. `heapq.merge` | O(n) | O(n) (slice) | tận dụng thư viện |

Trên thực tế, với `n ≤ 10^4`, Cách B chạy rất nhanh vì `sorted` viết bằng C — nhưng phỏng vấn sẽ hỏi Cách A.

## 11. Bộ test

In [13]:
tests = [
    ([-4, -1, 0, 3, 10], [0, 1, 9, 16, 100]),
    ([-7, -3, 2, 3, 11], [4, 9, 9, 49, 121]),
    ([5], [25]),                              # một phần tử
    ([-5], [25]),
    ([1, 2, 3], [1, 4, 9]),                   # toàn dương
    ([-3, -2, -1], [1, 4, 9]),                # toàn âm
    ([0, 0, 0], [0, 0, 0]),
    ([-2, -2, 2, 2], [4, 4, 4, 4]),           # trị tuyệt đối trùng
    ([-10000, 10000], [10**8, 10**8]),        # biên giá trị
]
impls = {
    "sort": sorted_squares_sort, "chính": sorted_squares,
    "LC-A": Solution().sortedSquares, "LC-B": SolutionSort().sortedSquares,
    "LC-C": SolutionMerge().sortedSquares, "LC-D": SolutionHeapMerge().sortedSquares,
}
for nums, expected in tests:
    for name, f in impls.items():
        got = f(nums)
        assert got == expected, (name, nums, got, expected)
    print(f"{str(nums):24} -> {expected} PASS")
print("Tất cả test PASS!")

[-4, -1, 0, 3, 10]       -> [0, 1, 9, 16, 100] PASS
[-7, -3, 2, 3, 11]       -> [4, 9, 9, 49, 121] PASS
[5]                      -> [25] PASS
[-5]                     -> [25] PASS
[1, 2, 3]                -> [1, 4, 9] PASS
[-3, -2, -1]             -> [1, 4, 9] PASS
[0, 0, 0]                -> [0, 0, 0] PASS
[-2, -2, 2, 2]           -> [4, 4, 4, 4] PASS
[-10000, 10000]          -> [100000000, 100000000] PASS
Tất cả test PASS!


In [14]:
import random
rng = random.Random(977)
for _ in range(2000):
    nums = sorted(rng.randint(-20, 20) for _ in range(rng.randint(1, 12)))
    expected = sorted(x * x for x in nums)
    for name, f in impls.items():
        assert f(nums) == expected, (name, nums)
print("2000 mảng ngẫu nhiên: mọi cách khớp")

2000 mảng ngẫu nhiên: mọi cách khớp


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ bình phương mà quên sắp xếp

In [15]:
def wrong_no_sort(nums):
    return [x * x for x in nums]

print("Sai :", wrong_no_sort([-4, -1, 0, 3, 10]))

Sai : [16, 1, 0, 9, 100]


**Vì sao sai:** số âm có bình phương lớn → thứ tự bị phá. **Sửa:** sắp xếp hoặc dùng hai con trỏ.

### Lỗi 2: Điền từ đầu với hai con trỏ ở hai đầu

In [16]:
def wrong_fill_front(nums):
    res, left, right = [], 0, len(nums) - 1
    while left <= right:
        if abs(nums[left]) < abs(nums[right]):     # chọn nhỏ hơn, điền từ đầu
            res.append(nums[left] ** 2); left += 1
        else:
            res.append(nums[right] ** 2); right -= 1
    return res

print("Sai :", wrong_fill_front([-4, -1, 0, 3, 10]))
print("Đúng:", sorted_squares([-4, -1, 0, 3, 10]))

Sai : [16, 1, 0, 9, 100]
Đúng: [0, 1, 9, 16, 100]


**Vì sao sai:** ở hai đầu ta chỉ biết **phần tử lớn nhất**, không biết phần tử nhỏ nhất (nó ở giữa). **Sửa:** điền từ **cuối** và chọn bên **lớn hơn**.

### Lỗi 3: Điều kiện `while left < right` (thiếu dấu `=`)

In [17]:
def wrong_lt(nums):
    n = len(nums)
    res = [0] * n
    left, right, pos = 0, n - 1, n - 1
    while left < right:
        if abs(nums[left]) > abs(nums[right]):
            res[pos] = nums[left] ** 2; left += 1
        else:
            res[pos] = nums[right] ** 2; right -= 1
        pos -= 1
    return res

print("Sai :", wrong_lt([-3, 1, 2]))
print("Đúng:", sorted_squares([-3, 1, 2]))

Sai : [0, 4, 9]
Đúng: [1, 4, 9]


**Vì sao sai:** khi `left == right` vẫn còn **một** phần tử chưa điền; vòng lặp dừng sớm nên `res[0]` giữ giá trị 0 mặc định. **Sửa:** `while left <= right` (hoặc dùng `for pos in range(n-1, -1, -1)`).

### Lỗi 4: So `nums[left] > nums[right]` thay vì trị tuyệt đối

`-4 > 3` là `False` nhưng `16 > 9`. Phải so `abs(...)` (hoặc so bình phương).

### Lỗi 5: Dùng `res.insert(0, x)` để điền từ đầu

Mỗi lần `insert(0, …)` phải dời toàn bộ phần tử → O(n) mỗi lần → tổng O(n²). Dùng mảng cố định + chỉ số `pos`, hoặc `append` rồi `reverse()` một lần.

## 13. Mẹo nhận dạng pattern

- Mảng **đã sắp xếp** + một phép biến đổi phá thứ tự kiểu chữ V (bình phương, trị tuyệt đối, khoảng cách tới một điểm) → **hai con trỏ ở hai đầu**, điền từ phía "cực trị" biết chắc.
- Kết quả cần sắp mà đầu vào gồm **hai dãy đã sắp** → **merge**.
- Đề có "Follow up: O(n)" với mảng đã sắp → gần như chắc chắn là two pointers.

## 14. Biến thể

1. **Sort Transformed Array (LeetCode 360, Premium):** áp `f(x) = ax² + bx + c` lên mảng đã sắp; parabol mở lên/xuống → điền từ cuối/đầu.
2. **LeetCode 88 — Merge Sorted Array:** trộn hai mảng đã sắp, điền từ cuối để làm tại chỗ.
3. **LeetCode 658 — Find K Closest Elements:** khoảng cách tới `x` cũng có hình chữ V → hai con trỏ thu hẹp.
4. Trả về bình phương **giảm dần** → điền từ đầu với phần tử lớn nhất.

In [18]:
def sort_transformed(nums, a, b_, c):
    f = lambda x: a * x * x + b_ * x + c
    n = len(nums)
    res = [0] * n
    l, r = 0, n - 1
    if a >= 0:                      # parabol mở lên: lớn nhất ở hai đầu -> điền từ cuối
        for pos in range(n - 1, -1, -1):
            if f(nums[l]) > f(nums[r]):
                res[pos] = f(nums[l]); l += 1
            else:
                res[pos] = f(nums[r]); r -= 1
    else:                           # mở xuống: nhỏ nhất ở hai đầu -> điền từ đầu
        for pos in range(n):
            if f(nums[l]) < f(nums[r]):
                res[pos] = f(nums[l]); l += 1
            else:
                res[pos] = f(nums[r]); r -= 1
    return res

print(sort_transformed([-4, -2, 2, 4], 1, 3, 5))
print(sort_transformed([-4, -2, 2, 4], -1, 3, 5))

[3, 9, 15, 33]
[-23, -5, 1, 7]


`lambda x: ...` tạo **hàm vô danh** một dòng — tiện để định nghĩa nhanh `f`.

## 15. Bài tập tự luyện (TODO)

### Bài 1 — Bình phương giảm dần
Trả về bình phương sắp **giảm dần**, O(n), không dùng `sorted`.

In [19]:
def squares_desc(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert squares_desc([-4, -1, 0, 3, 10]) == [100, 16, 9, 1, 0]

### Bài 2 — Trị tuyệt đối đã sắp
Trả về `[abs(x) ...]` sắp tăng dần, O(n).

In [20]:
def sorted_abs(nums: list[int]) -> list[int]:
    # TODO
    pass

# assert sorted_abs([-5, -3, 0, 2, 4]) == [0, 2, 3, 4, 5]

### Bài 3 — Số giá trị bình phương khác nhau
Đếm số giá trị **khác nhau** trong các bình phương, O(n) thời gian và O(1) bộ nhớ (dùng hai con trỏ).

In [21]:
def count_distinct_squares(nums: list[int]) -> int:
    # TODO
    pass

# assert count_distinct_squares([-3, -2, -1, 0, 1, 2]) == 4
# assert count_distinct_squares([-1, -1, 1, 1]) == 1

### Bài 4 — Merge Sorted Array
Cho hai list đã sắp `a`, `b`, trả về list sắp chứa mọi phần tử, O(len(a) + len(b)), không dùng `sorted`.

In [22]:
def merge_two(a: list[int], b: list[int]) -> list[int]:
    # TODO
    pass

# assert merge_two([1, 4, 7], [2, 3, 9, 10]) == [1, 2, 3, 4, 7, 9, 10]
# assert merge_two([], [1]) == [1]

### Bài 5 — K phần tử có bình phương nhỏ nhất
Trả về list `k` bình phương nhỏ nhất (tăng dần), O(log n + k), dùng `bisect_left` tìm điểm bắt đầu.

In [23]:
def k_smallest_squares(nums: list[int], k: int) -> list[int]:
    # TODO
    pass

# assert k_smallest_squares([-4, -1, 0, 3, 10], 3) == [0, 1, 9]
# assert k_smallest_squares([-7, -3, 2, 3, 11], 2) == [4, 9]

### Bài 6 (nâng cao) — Sort Transformed Array
Viết lại `sort_transformed(nums, a, b, c)` O(n) mà không nhìn mục 14.

In [24]:
def my_sort_transformed(nums: list[int], a: int, b: int, c: int) -> list[int]:
    # TODO
    pass

# assert my_sort_transformed([-4, -2, 2, 4], 1, 3, 5) == [3, 9, 15, 33]
# assert my_sort_transformed([-4, -2, 2, 4], -1, 3, 5) == [-23, -5, 1, 7]

## 16. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | lớn nhất ở hai đầu | giờ điền từ **đầu** | dùng `append` thay vì `res[pos]` |
| 2 | giống bài chính | thay `x*x` bằng `abs(x)` | điền từ cuối |
| 3 | hai con trỏ từ hai đầu | đếm mỗi khi giá trị lớn nhất mới khác giá trị trước | bỏ qua mọi phần tử có cùng trị tuyệt đối ở cả hai đầu |
| 4 | hai chỉ số `i`, `j` | lấy phần nhỏ hơn | nối phần còn lại sau vòng lặp |
| 5 | điểm đổi dấu `k` | `i = k − 1`, `j = k` | trộn ra hai phía, dừng sau `k` phần tử |
| 6 | dấu của `a` quyết định | `a ≥ 0`: điền từ cuối, lấy lớn | `a < 0`: điền từ đầu, lấy nhỏ |

## 17. Đáp án tham khảo

In [25]:
def ans_squares_desc(nums):
    l, r, res = 0, len(nums) - 1, []
    while l <= r:
        if abs(nums[l]) > abs(nums[r]):
            res.append(nums[l] ** 2); l += 1
        else:
            res.append(nums[r] ** 2); r -= 1
    return res

def ans_sorted_abs(nums):
    n = len(nums)
    res = [0] * n
    l, r = 0, n - 1
    for pos in range(n - 1, -1, -1):
        if abs(nums[l]) > abs(nums[r]):
            res[pos] = abs(nums[l]); l += 1
        else:
            res[pos] = abs(nums[r]); r -= 1
    return res

def ans_count_distinct_squares(nums):
    l, r = 0, len(nums) - 1
    count = 0
    while l <= r:
        cur = max(abs(nums[l]), abs(nums[r]))
        count += 1
        while l <= r and abs(nums[l]) == cur:
            l += 1
        while l <= r and abs(nums[r]) == cur:
            r -= 1
    return count

def ans_merge_two(a, b):
    i = j = 0
    res = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            res.append(a[i]); i += 1
        else:
            res.append(b[j]); j += 1
    res.extend(a[i:])
    res.extend(b[j:])
    return res

def ans_k_smallest_squares(nums, k):
    j = bisect_left(nums, 0)
    i = j - 1
    res = []
    while len(res) < k:
        if j >= len(nums) or (i >= 0 and -nums[i] < nums[j]):
            res.append(nums[i] ** 2); i -= 1
        else:
            res.append(nums[j] ** 2); j += 1
    return res

def ans_sort_transformed(nums, a, b, c):
    f = lambda x: a * x * x + b * x + c
    n = len(nums)
    res = [0] * n
    l, r = 0, n - 1
    positions = range(n - 1, -1, -1) if a >= 0 else range(n)
    for pos in positions:
        fl, fr = f(nums[l]), f(nums[r])
        take_left = fl > fr if a >= 0 else fl < fr
        if take_left:
            res[pos] = fl; l += 1
        else:
            res[pos] = fr; r -= 1
    return res

assert ans_squares_desc([-4, -1, 0, 3, 10]) == [100, 16, 9, 1, 0]
assert ans_sorted_abs([-5, -3, 0, 2, 4]) == [0, 2, 3, 4, 5]
assert ans_count_distinct_squares([-3, -2, -1, 0, 1, 2]) == 4
assert ans_count_distinct_squares([-1, -1, 1, 1]) == 1
assert ans_merge_two([1, 4, 7], [2, 3, 9, 10]) == [1, 2, 3, 4, 7, 9, 10]
assert ans_merge_two([], [1]) == [1]
assert ans_k_smallest_squares([-4, -1, 0, 3, 10], 3) == [0, 1, 9]
assert ans_k_smallest_squares([-7, -3, 2, 3, 11], 2) == [4, 9]
assert ans_sort_transformed([-4, -2, 2, 4], 1, 3, 5) == [3, 9, 15, 33]
assert ans_sort_transformed([-4, -2, 2, 4], -1, 3, 5) == [-23, -5, 1, 7]
for _ in range(500):
    arr = sorted(rng.randint(-9, 9) for _ in range(rng.randint(1, 10)))
    assert ans_count_distinct_squares(arr) == len({x * x for x in arr})
    k = rng.randint(1, len(arr))
    assert ans_k_smallest_squares(arr, k) == sorted(x * x for x in arr)[:k]
print("Đáp án: mọi test PASS!")

Đáp án: mọi test PASS!


## 18. Template nhớ nhanh + checklist submit

```python
class Solution:
    def sortedSquares(self, nums: list[int]) -> list[int]:
        n = len(nums)
        res = [0] * n
        l, r = 0, n - 1
        for pos in range(n - 1, -1, -1):        # điền từ CUỐI
            if abs(nums[l]) > abs(nums[r]):      # chọn bên có |x| LỚN hơn
                res[pos] = nums[l] * nums[l]; l += 1
            else:
                res[pos] = nums[r] * nums[r]; r -= 1
        return res
```

Câu thần chú: **"Lớn nhất nằm ở hai đầu → điền từ cuối."**

**Checklist trước khi submit**
- [ ] So trị tuyệt đối (hoặc bình phương), không so giá trị có dấu?
- [ ] Điền từ cuối, chọn bên lớn hơn?
- [ ] Vòng lặp xử lý cả phần tử cuối cùng (`<=` hoặc `for` đủ `n` lần)?
- [ ] Đã thử mảng toàn âm, toàn dương, một phần tử?
- [ ] Không dùng `insert(0, …)` gây O(n²)?